# Preliminary model tests

Here, we just want to check if it is possible to infer something with random weights on these models. The goal is validating the initial implementation before running training runs.

## Imports

In [1]:
import yaml

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from energy_transformers.model_config import TransformerConfig
from energy_transformers.baseline_transformer import RecursiveCGPT
from energy_transformers.energy_transformer import RecursiveNRGPT

In [2]:
# Load data
X = pd.read_parquet('../data/processed/listops32_tk_train_X.parquet')
y = pd.read_parquet('../data/processed/listops32_tk_train_y.parquet')
with open('../data/processed/listops32_dictionary.json', 'r') as f:
    dictionary = yaml.safe_load(f)

## Model and training config

In [3]:
seqlen = X.shape[1]
embed_dim = 64
n_head = 4
head_size = embed_dim // n_head
dropout=0.2
ff_hid_factor=4
vocab_size = len(dictionary)
n_layers = 1
n_classes = len(np.unique(y))

# Setting configurations
config = TransformerConfig(
    sequence_len=seqlen,
    n_embed=embed_dim,
    head_size=head_size,
    n_head=n_head,
    dropout=dropout,
    ff_hid_factor=ff_hid_factor,
    vocab_size=vocab_size,
    n_layers=n_layers,
    n_classes=n_classes,
    masked_attention=True
)

In [4]:
# Setting up training parameters
device = 'cuda' if torch.cuda.is_available() else 'cpu'

trainparam_lr = 3e-4
trainparam_batchsize = 64
trainparam_epochs = 1
trainparam_adamw_weight_decay = 1e-2
trainparam_scheduler_Tmax = 1000

In [5]:
# Setting um Dataset and DataLoaders
class ListOps32Dataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.long)
        self.y = torch.tensor(y.values, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
train_dataset = ListOps32Dataset(X, y)
train_loader = torch.utils.data.DataLoader(
    train_dataset, 
    batch_size=trainparam_batchsize, 
    shuffle=True
)

In [6]:
# Training function
def train_epoch(model, dataloader, optimizer, lr_scheduler):
    # Enable training
    model.train()

    # TODO: dont be stupid and actually make accumulative
    # average and std (first is easy, im lazy to think on the 2nd one)
    loss_history = np.zeros(len(dataloader))
    for batch_idx, (X,y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        y = y.squeeze()  # Remove extra dimension if present

        # Forward pass
        logits = model(X)
        loss = torch.nn.functional.cross_entropy(logits, y)

        # Backward pass and optimization
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        # Clip gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        lr_scheduler.step()

        # Save for stats
        loss_history[batch_idx] = loss.item()

        if batch_idx % 100 == 0:
            print(f"Batch {batch_idx}/{len(dataloader)}, Loss: {loss.item():.4f}")
    
    return loss_history.mean(), loss_history.std()

## Base Transformer model

In [7]:
# Trying out the Transformer block class
model_baseline = RecursiveCGPT(config)
print(f'Model parameters: {sum(p.numel() for p in model_baseline.parameters())}')

Model parameters: 54154


In [12]:
# Test training run

# Optimzer and scheduler
optimizer = torch.optim.AdamW(
    model_baseline.parameters(), 
    lr=trainparam_lr, 
    weight_decay=trainparam_adamw_weight_decay
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=trainparam_scheduler_Tmax
)

# Loop
loss_avg = []
loss_std = []
for epoch in tqdm(range(trainparam_epochs)):
    loss_epoch_avg, loss_epoch_std = train_epoch(
        model_baseline, 
        train_loader, 
        optimizer, 
        scheduler
    )
    loss_avg.append(loss_epoch_avg)
    loss_std.append(loss_epoch_std)

  0%|          | 0/1 [00:00<?, ?it/s]

Batch 0/1500, Loss: 1.6347
Batch 100/1500, Loss: 1.6822
Batch 200/1500, Loss: 1.7174
Batch 300/1500, Loss: 1.6668
Batch 400/1500, Loss: 1.7794
Batch 500/1500, Loss: 1.7700
Batch 600/1500, Loss: 1.8308
Batch 700/1500, Loss: 1.5330
Batch 800/1500, Loss: 1.4054
Batch 900/1500, Loss: 1.5789
Batch 1000/1500, Loss: 1.4652
Batch 1100/1500, Loss: 1.5901
Batch 1200/1500, Loss: 1.5344
Batch 1300/1500, Loss: 1.5249
Batch 1400/1500, Loss: 1.5120


100%|██████████| 1/1 [01:47<00:00, 107.85s/it]


## NRGPT

In [7]:
model_energy = RecursiveNRGPT(config)
print(f'Model parameters: {sum(p.numel() for p in model_energy.parameters())}')

Model parameters: 41611


In [8]:
# Test training run

# Optimzer and scheduler
optimizer = torch.optim.AdamW(
    model_energy.parameters(), 
    lr=trainparam_lr, 
    weight_decay=trainparam_adamw_weight_decay
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=trainparam_scheduler_Tmax
)

# Loop
loss_avg = []
loss_std = []
for epoch in tqdm(range(trainparam_epochs)):
    loss_epoch_avg, loss_epoch_std = train_epoch(
        model_energy, 
        train_loader, 
        optimizer, 
        scheduler
    )
    loss_avg.append(loss_epoch_avg)
    loss_std.append(loss_epoch_std)

  0%|          | 0/1 [00:00<?, ?it/s]

Batch 0/1500, Loss: 2.3374
Batch 100/1500, Loss: 1.8496
Batch 200/1500, Loss: 1.6439
Batch 300/1500, Loss: 1.7278
Batch 400/1500, Loss: 1.8131
Batch 500/1500, Loss: 1.7129
Batch 600/1500, Loss: 1.6471
Batch 700/1500, Loss: 1.9090
Batch 800/1500, Loss: 1.5312
Batch 900/1500, Loss: 1.4310
Batch 1000/1500, Loss: 1.5744
Batch 1100/1500, Loss: 1.6296
Batch 1200/1500, Loss: 1.6684
Batch 1300/1500, Loss: 1.7019
Batch 1400/1500, Loss: 1.6216


100%|██████████| 1/1 [03:54<00:00, 234.07s/it]
